In [0]:
# Read GCP credentials from Databricks secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Configure Spark to access GCS
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
import json

reviews_rdd = sc.textFile("gs://amazon_appliance/processed/reviews_clean/").map(json.loads)
meta_rdd = sc.textFile("gs://amazon_appliance/processed/meta_clean/").map(json.loads)

reviews_count = reviews_rdd.count()
print("cleaned reviews:", reviews_count)
print("cleaned products:", meta_rdd.count())

In [0]:
# (parent_asin, record) pairs for the join
reviews_kv = reviews_rdd.map(lambda x: (x["parent_asin"], x))
meta_kv = meta_rdd.map(lambda x: (x["parent_asin"], x))

# inner join on parent_asin (one product record per parent_asin)
joined_kv = reviews_kv.join(meta_kv)

def combine(x):
    parent_asin, (review, product) = x
    return {
        "parent_asin": parent_asin,
        "asin": review["asin"],
        "rating": review["rating"],
        "review_title": review["title"],
        "text": review["text"],
        "timestamp": review["timestamp"],
        "helpful_vote": review["helpful_vote"],
        "verified_purchase": review["verified_purchase"],
        "product_title": product["title"],
        "store": product["store"],
        "price": product["price"],
        "average_rating": product["average_rating"]
    }

joined_rdd = joined_kv.map(combine).cache()

joined_count = joined_rdd.count()
print("joined records:", joined_count)
print("reviews without product metadata:", reviews_count - joined_count)
print("fields:", list(joined_rdd.first().keys()))

In [0]:
output_path = "gs://amazon_appliance/processed/joined/"

# delete old output so the notebook can be rerun
out = sc._jvm.org.apache.hadoop.fs.Path(output_path)
fs = out.getFileSystem(hconf)
if fs.exists(out):
    fs.delete(out, True)

joined_rdd.map(json.dumps).saveAsTextFile(output_path)
print("joined data saved successfully")

In [0]:
saved_rdd = sc.textFile(output_path).map(json.loads)
saved_count = saved_rdd.count()

print("saved joined records:", saved_count)
print("missing join keys:", saved_rdd.filter(lambda x: not x["parent_asin"]).count())
print("sample:", saved_rdd.take(1))

assert saved_count == joined_count, "Saved count does not match joined count"
print("validation successful")